In [1]:
froot = r"D:\pAce\BKV009\20260512\FOV1_T14"
analysis_mode = "new"  # or "cellpose" for cell segmentation
folder_paths = [froot]

In [2]:

print("Importing packages and Initializing...")
version="V1"

from pathlib import Path
current_dir = r"C:\Users\strenglab\VoImAn\caiman\ICNLAB"
weights_path= str(current_dir + "//" + "mask_rcnn_neuron_0012.h5")

#V1.2: 0.8 corr cutoff, 2 minimum ratio of h over w for spikes, cell_idxs incremented by 1, wheel data appended to mat save
print("version:", version)

import matplotlib
matplotlib.use("QtAgg")   # interactive, no windows
print(matplotlib.get_backend())

from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from PIL import Image
import re
import csv
from datetime import datetime

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd


from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc
from caiman.ICNLAB.single_trial_simple_plotting import plotdata

logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)


Importing packages and Initializing...
version: V1
QtAgg


In [3]:

##BEGIN MAIN ANALYSIS LOOP
folder_path = folder_paths[0]  # Select the first folder path

#print to log even if no tsm
rootpath = Path(folder_path).parts[0] + Path(folder_path).parts[-4] + '\\Analysis\\'
Path(rootpath).mkdir(parents=True, exist_ok=True)
log_csv_path = Path(rootpath) / "MasterAnalysisLOG.csv" #Master CSV path
unique_save_string1 = "-".join(Path(folder_path).parts[-3:])

# find the .tsm file in the folder
tsm_files = [f for f in os.listdir(folder_path) if f.endswith(('.tsm', '.dcimg'))]
if not tsm_files:
    print(f"No recording files found in {folder_path}, skipping.")
    #Append new row to MASTERLOG
    today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
    new_row = [version, today_str, unique_save_string1, "No recording file"]
    with open(log_csv_path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(new_row)
    print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")
#continue if more than one .tsm file found
if len(tsm_files) > 1:
    print(f"Multiple recording files found in {folder_path}, skipping.")
    #Append new row to MASTERLOG
    today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
    new_row = [version, today_str, unique_save_string1, "Multiple recording files"]
    with open(log_csv_path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(new_row)
    print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")



fname = os.path.join(folder_path, tsm_files[0])
print('fname is', fname)
print("Processing file:", fname)



fname is D:\pAce\BKV009\20260512\FOV1_T14\FOV1_T14_Green.dcimg
Processing file: D:\pAce\BKV009\20260512\FOV1_T14\FOV1_T14_Green.dcimg


In [20]:


fpath = Path(fname)
#Create new unique save name
unique_save_string = "-".join(fpath.parts[-4:-1])
rootpath = str(Path(*fpath.parts[:-4]))+'\\Analysis\\'
print("Unique save string:", unique_save_string)
print("Directory for Analysis Files:", rootpath)
Path(rootpath).mkdir(parents=True, exist_ok=True)
log_csv_path = Path(rootpath) / "MasterAnalysisLOG.csv" #Master CSV path

#grab data for plotting with single_trial_simple_plotting.py
mouseID = fpath.parts[-4]
date = fpath.parts[-3]
trialname = fpath.parts[-2]

##
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'
fr = 3600  ################################################################REMOVE LATER\
H, W = 1108,18
print(fname, fr)


##
# Cleanup R:/ drive (temp RAM disk)
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")


##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

##
print("Loading data...")
m_orig = cm.load(fname)
ds_ratio = 0.2

##
try:
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
except:
    print("Cluster running doing restart")
    dview.terminate()
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
    
##
print("Motion correction...")
mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True, save_dir="R:/")
#about 2.3 minutes for 12800 frames (2m 13-21 s)
print("Done.")

##
print("Loading corrected movie...")
m_rig = cm.load(mc.mmap_file) # 11s
ds_ratio = 0.2
print("Done.")

del m_orig
gc.collect()

####CONVERT VIA STREAMING WITH HOPEFULLY SAME OG BEHAVIOR
p = Path(fname)

ram_path = Path(r'R:/') / (
    f"{p.stem}_rig__d1_{m_rig.shape[1]}"
    f"_d2_{m_rig.shape[2]}"
    f"_d3_1_order_C_frames_{m_rig.shape[0]}.mmap"
)
ram_path = str(ram_path).replace("/", "\\")

# Destination memmap: SAME AS ORIGINAL
dst = np.memmap(
    ram_path,
    dtype='float32',
    mode='w+',
    shape=m_rig.shape,
    order='F'   # critical: this is what caused the layout change originally
)

# Streaming copy (logical copy, not byte copy)
chunk = 16  # frames per chunk; tune for cache / IO

T = m_rig.shape[0]

for t0 in range(0, T, chunk):
    t1 = min(t0 + chunk, T)
    dst[t0:t1] = m_rig[t0:t1]

dst.flush()
mmap_list = [dst]

if hasattr(dst, 'base') and hasattr(dst.base, 'close'):
    dst.base.close()
del dst
gc.collect()

##


Unique save string: BKV009-20260512-FOV1_T14
Directory for Analysis Files: D:\pAce\Analysis\
D:\pAce\BKV009\20260512\FOV1_T14\FOV1_T14_Green.dcimg 3600
Cleaning up R:/ drive...
Cleared all files from R:/
Loading data...
Cluster running doing restart
Motion correction...
Saving mmap to:  R:/FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_F_frames_36000.mmap
Done.
Loading corrected movie...


100%|██████████| 1/1 [00:01<00:00,  1.97s/it]


Done.


0

In [38]:

print("Computing mean and correlation images...")
img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)

#Display img
plt.figure(figsize=(15, 2))
plt.imshow(img, cmap="viridis")
plt.colorbar()
plt.show()
print(img.shape)

Computing mean and correlation images...
(18, 1108)


In [ ]:

# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.signal import butter, filtfilt
# from tqdm import tqdm

# # ===============================
# # 1. Parameters
# # ===============================
# HIGHPASS_THRESH = (5)
# shape = m_rig.shape
# # ===============================
# # 2. Load memory-mapped video
# # ===============================

# video = np.memmap(
#     mc.mmap_file[0],
#     dtype=np.float32,
#     mode="r",
#     shape=shape,
#     order="C"
# ).swapaxes(1, 2)

# T, H, W = video.shape
# print(f"Loaded video: {video.shape}")

# # ===============================
# # 3. High-pass filter (bandpass-compatible API)
# # ===============================
# def highpass_filter(data, fs, low, high=None, order=3):
#     """
#     High-pass filter using the 'low' cutoff.
#     The 'high' argument is accepted for API compatibility but ignored.
#     """
#     nyq = 0.5 * fs
#     b, a = butter(order, low / nyq, btype="high")
#     return filtfilt(b, a, data, axis=0)


# # ===============================
# # Parameters
# # ===============================
# TILE_SIZE = 2
# H, W = H, W
# FRAME_RATE = fr
# # (low, high), high ignored
# DISPLAY_CLIP = 99

# # ===============================
# # Coherence metric
# # ===============================
# def coherence_metric(tile_filt):
#     """
#     tile_filt: shape (T, Npix)
#     Returns mean pixel-to-tile correlation.
#     """
#     # Tile reference (subthreshold signals sum coherently)
#     ref = tile_filt.mean(axis=1)


#     ref -= ref.mean()
#     ref_std = ref.std() + 1e-9

#     # Normalize reference
#     ref /= ref_std

#     # Normalize pixels
#     pix = tile_filt - tile_filt.mean(axis=0)
#     pix /= (pix.std(axis=0) + 1e-9)

#     # Correlation with reference
#     corr = np.mean(ref[:, None] * pix, axis=0)

#     # Use mean absolute correlation as coherence
#     return np.mean(np.abs(corr))


# # ===============================
# # Output tile map
# # ===============================
# n_tiles_y = H // TILE_SIZE
# n_tiles_x = W // TILE_SIZE

# tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# # ===============================
# # Main loop
# # ===============================
# with tqdm(total=n_tiles_y * n_tiles_x, desc="Computing coherence") as pbar:
#     for ty in range(n_tiles_y):
#         for tx in range(n_tiles_x):

#             y0 = ty * TILE_SIZE
#             y1 = y0 + TILE_SIZE
#             x0 = tx * TILE_SIZE
#             x1 = x0 + TILE_SIZE

#             # Extract tile: (T, 16, 16)
#             tile = video[:, y0:y1, x0:x1]
#             tile = tile.reshape(T, -1)

#             # High-pass filter all pixels independently
#             tile_filt = highpass_filter(
#                 tile, FRAME_RATE, HIGHPASS_THRESH
#             )

#             # Compute coherence
#             tile_coherence_map[ty, tx] = coherence_metric(tile_filt)

#             pbar.update(1)

# # ===============================
# # Expand to image resolution
# # ===============================
# coherence_image = np.repeat(
#     np.repeat(tile_coherence_map, TILE_SIZE, axis=0),
#     TILE_SIZE, axis=1
# )

# if hasattr(video, 'base') and hasattr(video.base, 'close'):
#     video.base.close()

# del video
# gc.collect()


In [ ]:
#NEW STRENG LAB VERSION CHANGED LOADING TO ORDER F AND SOS HIGH PASS FILTERING
#DEBUG LOOP WITH 3 TEST CUT OFFS
#  import numpy as np
# import matplotlib.pyplot as plt
# from scipy.signal import butter, sosfiltfilt
# from tqdm import tqdm
# import gc

# # ===============================
# # 1. Configuration & Parameters
# # ===============================
# # We assume m_rig.shape is (T, H, W), e.g., (36000, 18, 1108)
# T_frames, H_dim, W_dim = m_rig.shape 
# FRAME_RATE = 3600
# TILE_SIZE = 2
# TEST_CUTOFFS = [2.0, 5.0, 10.0]

# # ===============================
# # 2. Load Memory-Mapped Video
# # ===============================
# print("Loading memory-mapped file...")

# # Load the file in the exact way CaImAn wrote it to disk: (H, W, T) and order="F"
# video_raw = np.memmap(
#     mc.mmap_file[0],
#     dtype=np.float32,
#     mode="r",
#     shape=(H_dim, W_dim, T_frames),
#     order="F"
# )

# # Transpose the axes to get it back to Python's preferred (Time, Height, Width)
# video = np.transpose(video_raw, (2, 0, 1))

# print(f"Correctly oriented video: {video.shape}")

# # ===============================
# # 3. Core Logic Function
# # ===============================
# def compute_coherence_map(vid_data, fs, cutoff, tile_size):
#     t_frames, height, width = vid_data.shape
#     n_tiles_y = height // tile_size
#     n_tiles_x = width // tile_size
    
#     tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)
    
#     # Use SOS (Second-Order Sections) for numerical stability at high sampling rates
#     nyq = 0.5 * fs
#     sos = butter(3, cutoff / nyq, btype="high", output="sos")
    
#     with tqdm(total=n_tiles_y * n_tiles_x, desc=f"Processing {cutoff}Hz cutoff") as pbar:
#         for ty in range(n_tiles_y):
#             for tx in range(n_tiles_x):
#                 y0, y1 = ty * tile_size, (ty + 1) * tile_size
#                 x0, x1 = tx * tile_size, (tx + 1) * tile_size

#                 # Extract and flatten spatial dimensions: (T, Npix)
#                 tile = vid_data[:, y0:y1, x0:x1].reshape(t_frames, -1)

#                 # High-pass filter using stable SOS
#                 tile_filt = sosfiltfilt(sos, tile, axis=0)

#                 # Coherence metric calculation
#                 ref = tile_filt.mean(axis=1)
#                 ref -= ref.mean()
#                 ref /= (ref.std() + 1e-9)

#                 pix = tile_filt - tile_filt.mean(axis=0)
#                 pix /= (pix.std(axis=0) + 1e-9)

#                 corr = np.mean(ref[:, None] * pix, axis=0)
#                 tile_coherence_map[ty, tx] = np.mean(np.abs(corr))

#                 pbar.update(1)
                
#     # Expand to image resolution
#     expanded_map = np.repeat(
#         np.repeat(tile_coherence_map, tile_size, axis=0),
#         tile_size, axis=1
#     )
#     return expanded_map

# # ===============================
# # 4. Main Loop & Plotting
# # ===============================
# results = []
# for cutoff in TEST_CUTOFFS:
#     cmap = compute_coherence_map(video, FRAME_RATE, cutoff, TILE_SIZE)
#     results.append((cutoff, cmap))

# # Clean up memory map before plotting to free RAM
# print("Cleaning up memory...")
# if hasattr(video_raw, 'base') and hasattr(video_raw.base, 'close'):
#     video_raw.base.close()
# del video
# del video_raw
# gc.collect()

# # Plotting the results side-by-side
# print("Generating plots...")
# fig, axes = plt.subplots(1, len(TEST_CUTOFFS), figsize=(18, 8))

# # Ensure axes is iterable even if there's only 1 cutoff test
# if len(TEST_CUTOFFS) == 1:
#     axes = [axes]

# for ax, (cutoff, cmap_data) in zip(axes, results):
#     # Transpose the data here so it is 1108 tall and 18 wide
#     plot_data = cmap_data.T 
    
#     # Use robust scaling to prevent outliers from washing out the image
#     vmin = np.nanpercentile(plot_data, 2)
#     vmax = np.nanpercentile(plot_data, 98)
    
#     # Using aspect="equal" to keep true pixel proportions without stretching the 18-pixel width
#     im = ax.imshow(plot_data, cmap="viridis", aspect="equal", vmin=vmin, vmax=vmax)
#     ax.set_title(f"High-pass Cutoff: {cutoff} Hz")
    
#     # Add colorbar 
#     fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# plt.tight_layout()
# plt.show()

Loading memory-mapped file...
Correctly oriented video: (36000, 18, 1108)


Processing 10.0Hz cutoff: 100%|██████████| 4986/4986 [00:24<00:00, 200.92it/s]


Cleaning up memory...
Generating plots...


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from tqdm import tqdm
import gc

# ===============================
# 1. Configuration & Parameters
# ===============================
# We assume m_rig.shape is (T, H, W), e.g., (36000, 18, 1108)
T_frames, H_dim, W_dim = m_rig.shape 
FRAME_RATE = 3600
TILE_SIZE = 2
CUTOFF = 5.0  # Run only for 5Hz cutoff

# ===============================
# 2. Load Memory-Mapped Video
# ===============================
print("Loading memory-mapped file...")

# Load the file in the exact way CaImAn wrote it to disk: (H, W, T) and order="F"
video_raw = np.memmap(
    mc.mmap_file[0],
    dtype=np.float32,
    mode="r",
    shape=(H_dim, W_dim, T_frames),
    order="F"
)

# Transpose the axes to get it back to Python's preferred (Time, Height, Width)
video = np.transpose(video_raw, (2, 0, 1))

print(f"Correctly oriented video: {video.shape}")

# ===============================
# 3. Core Logic Function
# ===============================
def compute_coherence_map(vid_data, fs, cutoff, tile_size):
    t_frames, height, width = vid_data.shape
    n_tiles_y = height // tile_size
    n_tiles_x = width // tile_size
    
    tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)
    
    # Use SOS (Second-Order Sections) for numerical stability at high sampling rates
    nyq = 0.5 * fs
    sos = butter(3, cutoff / nyq, btype="high", output="sos")
    
    with tqdm(total=n_tiles_y * n_tiles_x, desc=f"Processing {cutoff}Hz cutoff") as pbar:
        for ty in range(n_tiles_y):
            for tx in range(n_tiles_x):
                y0, y1 = ty * tile_size, (ty + 1) * tile_size
                x0, x1 = tx * tile_size, (tx + 1) * tile_size

                # Extract and flatten spatial dimensions: (T, Npix)
                tile = vid_data[:, y0:y1, x0:x1].reshape(t_frames, -1)

                # High-pass filter using stable SOS
                tile_filt = sosfiltfilt(sos, tile, axis=0)

                # Coherence metric calculation
                ref = tile_filt.mean(axis=1)
                ref -= ref.mean()
                ref /= (ref.std() + 1e-9)

                pix = tile_filt - tile_filt.mean(axis=0)
                pix /= (pix.std(axis=0) + 1e-9)

                corr = np.mean(ref[:, None] * pix, axis=0)
                tile_coherence_map[ty, tx] = np.mean(np.abs(corr))

                pbar.update(1)
                
    # Expand to image resolution
    expanded_map = np.repeat(
        np.repeat(tile_coherence_map, tile_size, axis=0),
        tile_size, axis=1
    )
    return expanded_map

# ===============================
# 4. Main Run & Plotting
# ===============================
cmap_data = compute_coherence_map(video, FRAME_RATE, CUTOFF, TILE_SIZE)

# Clean up memory map before plotting to free RAM
print("Cleaning up memory...")
if hasattr(video_raw, 'base') and hasattr(video_raw.base, 'close'):
    video_raw.base.close()
del video
del video_raw
gc.collect()

# Plotting the result


Loading memory-mapped file...
Correctly oriented video: (36000, 18, 1108)


Processing 5.0Hz cutoff: 100%|██████████| 4986/4986 [00:25<00:00, 194.37it/s]


Cleaning up memory...
Generating plot...


In [37]:
print("Generating plot...")
plt.figure(figsize=(10, 8))

# Transpose the data here so it is 1108 tall and 18 wide
plot_data = cmap_data.T 

# Use robust scaling to prevent outliers from washing out the image
vmin = np.nanpercentile(cmap_data, 2)
vmax = np.nanpercentile(cmap_data, 98)

# Using aspect="equal" to keep true pixel proportions without stretching the 18-pixel width
im = plt.imshow(cmap_data, cmap="viridis", aspect="equal", vmin=vmin, vmax=vmax)
plt.title(f"High-pass Cutoff: {CUTOFF} Hz")

# Add colorbar 
plt.colorbar(im, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

Generating plot...


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np

# ---------------------------------------------------------
# 1. Print Image Size and Stats
# ---------------------------------------------------------
print(f"Coherence image shape: {coherence_image.shape}")
print(f"Total pixels: {coherence_image.size:,}")
# Using np.nanmin/nanmax just in case your data contains any NaN values
print(f"Min intensity: {np.nanmin(coherence_image):.4f}")
print(f"Max intensity: {np.nanmax(coherence_image):.4f}")
print(f"Mean intensity: {np.nanmean(coherence_image):.4f}")
print("-" * 30)

# ---------------------------------------------------------
# 2. Try Different Scalings side-by-side
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# Calculate "robust" limits (ignoring the bottom 1% and top 1% of extreme pixels)
robust_vmin = np.nanpercentile(coherence_image, 1)
robust_vmax = np.nanpercentile(coherence_image, 99)

# A. Robust Linear Scaling (Clips extreme outliers)
im0 = axes[0].imshow(coherence_image, cmap="viridis", aspect="auto", 
                     vmin=robust_vmin, vmax=robust_vmax)
axes[0].set_title("Robust Linear (1st-99th Percentile)")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# B. Logarithmic Scaling (Great if data has a massive dynamic range)
# Log scaling doesn't like zeros/negatives, so we add a tiny epsilon just in case
epsilon = 1e-6 
valid_log_min = max(robust_vmin, epsilon)
im1 = axes[1].imshow(coherence_image + epsilon, cmap="viridis", aspect="auto", 
                     norm=colors.LogNorm(vmin=valid_log_min, vmax=robust_vmax))
axes[1].set_title("Logarithmic Scale")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# C. Power-Law / Gamma Scaling (gamma < 1 boosts dark pixels, > 1 boosts brights)
im2 = axes[2].imshow(coherence_image, cmap="viridis", aspect="auto", 
                     norm=colors.PowerNorm(gamma=0.5, vmin=np.nanmin(coherence_image), vmax=robust_vmax))
axes[2].set_title("Power Norm (Gamma=0.5)")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 3. Plot the Histogram of Intensity Values
# ---------------------------------------------------------
plt.figure(figsize=(10, 5))

# .ravel() flattens the 11008x18 array into a single 1D list for the histogram
# We also filter out NaNs if any exist to prevent matplotlib errors
valid_pixels = coherence_image[~np.isnan(coherence_image)] 

# Plotting 100 bins. 
plt.hist(valid_pixels.ravel(), bins=100, color='teal', edgecolor='black', alpha=0.7)

plt.title("Histogram of Pixel Intensities")
plt.xlabel("Intensity Value")
plt.ylabel("Frequency (Number of Pixels)")

# We set the Y-axis to log scale. Because you have ~200,000 pixels, 
# a log scale prevents the tallest bar from squishing all the smaller bars into invisibility.
plt.yscale('log') 
plt.grid(axis='y', alpha=0.5)

plt.show()

Coherence image shape: (1108, 18)
Total pixels: 19,944
Min intensity: 0.4939
Max intensity: 0.5800
Mean intensity: 0.5230
------------------------------


In [9]:

# ===============================
# Visualization
# ===============================
vmax = np.percentile(coherence_image, DISPLAY_CLIP)

plt.figure(figsize=(6, 6))
plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax)
plt.title("Grid-based subthreshold coherence ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
plt.colorbar(label="Mean |pixel–tile correlation|")
plt.axis("off")
plt.tight_layout()
plt.show()



In [14]:
# Make the figure window significantly larger (width, height)
plt.figure(figsize=(15, 10)) 

# aspect="auto" stretches the 18 pixels so they are visible alongside the 11008
plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax, aspect="auto")

plt.colorbar() # Highly recommend adding this if you are using vmin/vmax!
plt.show()

In [41]:

img_corr = cmap_data
print(f"Coherence image shape: {cmap_data.shape}")
print(f" image shape: {img.shape}")


Coherence image shape: (18, 1108)
 image shape: (18, 1108)


In [42]:
img_corr = cmap_data

summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
#cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')

plt.imshow(summary_images[0], cmap='gray')
plt.axis('off')
#plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)


plt.imshow(summary_images[2], cmap='gray')
plt.axis('off')
#plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)

img = summary_images.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img.shape[:2]
print(img.shape)

# --------------------------------------------------------------
# Extract channels like MATLAB
# --------------------------------------------------------------
R = img[:, :, 0]
B = img[:, :, 2]

# --------------------------------------------------------------
# MATLAB-style normalization (mat2gray + uint8)
# --------------------------------------------------------------
def normalize_like_matlab(x):
    x = x.astype(np.float64)
    mn = x.min()
    mx = x.max()
    x = (x - mn) / (mx - mn + 1e-12)

    # MATLAB uint8 applies rounding, not floor
    x = np.round(255 * x).astype(np.uint8)
    return x

R_norm = normalize_like_matlab(R)
B_norm = normalize_like_matlab(B)

# --------------------------------------------------------------
# Build MATLAB-equivalent RGB (R,R,B)
# --------------------------------------------------------------
rgb = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)

# --------------------------------------------------------------
# Save as PNG/TIF (MATLAB-compatible pixel data)
# --------------------------------------------------------------
outname =  rootpath + unique_save_string + ".tif"
Image.fromarray(rgb).save(outname)

print("Saved:", outname)
img = rgb.copy()



D:\pAce\BKV009\20260512\FOV1_T14\FOV1_T14_Green.d_corr.tif
(18, 1108, 3)
Saved: D:\pAce\Analysis\BKV009-20260512-FOV1_T14.tif


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Create the 512x512 Canvas
# ---------------------------------------------------------
# Ensure canvas matches the data type of your image (likely uint8 or float32)
canvas_size = 512
canvas = np.zeros((canvas_size, canvas_size, 3), dtype=img.dtype)

h, w, c = img.shape
chunk_width = canvas_size - 20  # Leave a 10px margin on the left/right
overlap = 40                    # Number of pixels to overlap between strips
stride = chunk_width - overlap  # How far to step forward each time

y_start = 50                    # Start 50 pixels down from the top
y_spacing = 60                  # Spacing between strips

# ---------------------------------------------------------
# 2. Slice the 18x1108 image and stack it onto the canvas
# ---------------------------------------------------------
print(f"Tiling image of shape {img.shape} onto a 512x512 canvas with {overlap}px overlap...")

chunks = []
current_y = y_start

for x in range(0, w, stride):
    # Ensure we don't grab more pixels than are left in the image
    c_w = min(chunk_width, w - x)
    
    # Slice a chunk of the image
    chunk = img[:, x : x + c_w, :]
    
    # Paste it onto the canvas
    canvas[current_y : current_y + h, 10 : 10 + c_w, :] = chunk
    
    # Define the "core" x-region for this chunk to prevent double counting in overlaps.
    # The first chunk owns the first half of its overlap, the next chunk owns the second half.
    core_x_start = 10 + (overlap // 2 if x > 0 else 0)
    core_x_end = 10 + c_w - (overlap // 2 if x + stride < w else 0)
    
    # Save the coordinates to map detections back and filter invalid ones
    chunks.append({
        'orig_x_start': x,
        'canvas_y_start': current_y,
        'canvas_y_end': current_y + h,
        'canvas_x_start': 10,
        'canvas_x_end': 10 + c_w,
        'core_x_start': core_x_start,
        'core_x_end': core_x_end,
        'chunk_width': c_w
    })
    
    current_y += y_spacing

# ---------------------------------------------------------
# 3. Run Inference on the Composite Canvas
# ---------------------------------------------------------
print("Running Mask R-CNN inference on tiled canvas...")
r = utils.mrcnn_inference(canvas, size_range=[0, 40], weights_path=weights_path, display_result=False)

# ---------------------------------------------------------
# 4. Filter Invalid, Duplicate, and Transform Detections
# ---------------------------------------------------------
valid_orig_masks = []  # To hold the masks transformed back to 18x1108
valid_orig_rois = []   # To hold the bounding boxes transformed back
valid_scores = []
valid_class_ids = []

if r['masks'].shape[-1] > 0:
    n_detections = r['masks'].shape[-1]
    
    for i in range(n_detections):
        # r['rois'] is typically [y1, x1, y2, x2]
        y1, x1, y2, x2 = r['rois'][i]
        
        # Calculate the center point of the detection
        cy = (y1 + y2) / 2.0
        cx = (x1 + x2) / 2.0
        
        # Check if the center falls strictly inside the "core" region of any chunk
        is_valid = False
        assigned_chunk = None
        for chunk in chunks:
            if (chunk['canvas_y_start'] <= cy <= chunk['canvas_y_end']) and \
               (chunk['core_x_start'] <= cx <= chunk['core_x_end']):
                is_valid = True
                assigned_chunk = chunk
                break
                
        if is_valid:
            valid_scores.append(r['scores'][i])
            valid_class_ids.append(r['class_ids'][i])
            
            # --- Transform ROIs back to original coordinates ---
            orig_y1 = y1 - assigned_chunk['canvas_y_start']
            orig_y2 = y2 - assigned_chunk['canvas_y_start']
            orig_x1 = x1 - assigned_chunk['canvas_x_start'] + assigned_chunk['orig_x_start']
            orig_x2 = x2 - assigned_chunk['canvas_x_start'] + assigned_chunk['orig_x_start']
            
            # Clip to image boundaries to prevent out-of-bounds indices
            orig_y1 = max(0, min(h, orig_y1))
            orig_y2 = max(0, min(h, orig_y2))
            orig_x1 = max(0, min(w, orig_x1))
            orig_x2 = max(0, min(w, orig_x2))
            
            valid_orig_rois.append([orig_y1, orig_x1, orig_y2, orig_x2])
            
            # --- Transform Masks back to original coordinates ---
            # Extract the valid mask using the assigned chunk's canvas coordinates
            mask_chunk = r['masks'][
                assigned_chunk['canvas_y_start'] : assigned_chunk['canvas_y_end'],
                assigned_chunk['canvas_x_start'] : assigned_chunk['canvas_x_end'],
                i
            ]
            
            # Paste it into an empty mask matching the original image dimensions
            orig_mask = np.zeros((h, w), dtype=bool)
            orig_mask[:, assigned_chunk['orig_x_start'] : assigned_chunk['orig_x_start'] + assigned_chunk['chunk_width']] = mask_chunk
            valid_orig_masks.append(orig_mask)

# --- Compile the Final Reconstructed Dictionary ---
r_reconstructed = {
    'rois': np.array(valid_orig_rois, dtype=np.int32) if len(valid_orig_rois) > 0 else np.empty((0, 4), dtype=np.int32),
    'class_ids': np.array(valid_class_ids, dtype=np.int32),
    'scores': np.array(valid_scores, dtype=np.float32),
    'masks': np.stack(valid_orig_masks, axis=-1) if len(valid_orig_masks) > 0 else np.empty((h, w, 0), dtype=bool)
}

print("\n--- Reconstructed Results ---")
print("r_reconstructed keys:", r_reconstructed.keys())
print("r_reconstructed['rois'] shape:", r_reconstructed['rois'].shape)
print("r_reconstructed['masks'] shape:", r_reconstructed['masks'].shape)
print("r_reconstructed['scores']:", r_reconstructed['scores'])


# ---------------------------------------------------------
# 5. Summarize and Plot the Tiled Canvas Results
# ---------------------------------------------------------
# We will use the r_reconstructed object for our downstream checks to ensure it works!
n_valid = r_reconstructed['masks'].shape[-1]

if n_valid > 0:
    print(f"\nSuccess! Found {n_valid} valid objects strictly within the image regions.")
    # Show the canvas detections by projecting the reconstructed masks back onto a visual sum
    # (Optional: If you still wanted to see the canvas sum, we skip it here to focus on the real data)
else:
    print("\nNo valid objects detected in the continuous image regions.")

fig, axs = plt.subplots(1, 2, figsize=(14, 7))

# Show the canvas we built
axs[0].imshow(canvas)
axs[0].set_title('Composite 512x512 Input Canvas')
axs[0].axis('off')

# Show original image with reconstructed masks overlayed directly as a sum
axs[1].imshow(img, aspect="equal")  # Background
if n_valid > 0:
    masks_sum = r_reconstructed['masks'].sum(axis=-1)
    axs[1].imshow(masks_sum, cmap='jet', alpha=0.5, aspect="equal")  # Overlay valid masks
axs[1].set_title(f'Mask R-CNN Reconstructed Valid Masks (n={n_valid})')
axs[1].axis('off')

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 6. Reconstruct and Plot Original Image with Contours
# ---------------------------------------------------------
print("Generating reconstructed original image with contours...")

fig2, ax2 = plt.subplots(figsize=(18, 4))

# Show original image (aspect="equal" to maintain true pixel proportions)
ax2.imshow(img, aspect="equal") 
ax2.set_title(f'Original Image ({h}x{w}) Reconstructed with {n_valid} ROI Contours')
ax2.axis('off')

# Overlay each valid mask from our reconstructed dictionary as a contour line
if n_valid > 0:
    for i in range(n_valid):
        mask = r_reconstructed['masks'][:, :, i]
        if np.any(mask):
            ax2.contour(mask, levels=[0.5], colors='red', linewidths=1)

plt.tight_layout()
plt.show()


r=r_reconstructed
ROIs = r['masks'].transpose([2, 0, 1])
Coords = r['rois']
#cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')


fig, axs = plt.subplots(2, 1, figsize=(10, 15))
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')

Tiling image of shape (18, 1108, 3) onto a 512x512 canvas with 40px overlap...
Running Mask R-CNN inference on tiled canvas...

Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_c

Text(0.5, 1.0, 'masks')

In [71]:
print(r.keys())
print(r["rois"][0].shape)
print(r["class_ids"])
print(r["scores"])
print(r["masks"].shape)


dict_keys(['rois', 'class_ids', 'scores', 'masks'])
(4,)
[1 1 1 1 1 1 1 1 1 1 1]
[0.9991259  0.99695647 0.98956394 0.96333086 0.89797443 0.8515559
 0.8378921  0.830831   0.7480387  0.63238037 0.5974065 ]
(512, 512, 11)


In [72]:
print(r["rois"])

[[ 49 480  64 503]
 [110  28 126  44]
 [110  90 125 105]
 [113 113 125 129]
 [108 315 124 330]
 [ 50 422  65 434]
 [ 48 367  63 378]
 [107  10 119  21]
 [107 490 119 504]
 [ 51 291  66 307]
 [108  44 122  64]]


In [43]:


##
print("Running Mask R-CNN inference...")
#download_model('mask_rcnn')
#ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=False)
ROIs = r['masks'].transpose([2, 0, 1])
Coords = r['rois']
#cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')

fig, axs = plt.subplots(1, 2)
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')
#plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)
# Save the figure and close the plot   

#save ROIs as npy array
#np.save(fname[:-4]+'newmrcnn_ROIs.npy', ROIs)
#print("Saved ROIs as npy array:", fname[:-4]+'newmrcnn_ROIs.npy')


Running Mask R-CNN inference...

Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_class_loss': 1.0, 'mrcnn_bbox_loss': 1.0, 'mrcnn_mask_loss': 1.0}
MASK_POOL_SIZE                

     3799014 [deprecation.py:            new_func():554][14716] From c:\Users\strenglab\anaconda3\envs\caiman\lib\site-packages\tensorflow\python\util\deprecation.py:629: calling map_fn_v2 (from tensorflow.python.ops.map_fn) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Use fn_output_signature instead


Processing 1 images
image                    shape: (18, 1108, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min:  -77.11000  max:   19.24000  float64
image_metas              shape: (1, 14)               min:    0.00000  max: 1108.00000  float64
anchors                  shape: (1, 65472, 4)         min:   -0.04428  max:    1.01297  float32


c:\Users\strenglab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,



*** No instances to display *** 

MADE FIGURE


Text(0.5, 1.0, 'masks')

In [68]:
print(ram_path)

R:\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap


In [67]:
import numpy as np
import cv2
import os

# ---------------------------------------------------------
# 1. Configuration from your filename metadata
# ---------------------------------------------------------
# Path variable as requested
# ram_path = r"R:\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap"

d1 = 18       # Height
d2 = 1108     # Width
d3 = 1        # Channels
frames = 36000
order = 'F'   # C-style memory layout

# Expected shape for standard video frame indexing: (Frames, Height, Width)
video_shape = (frames, d1, d2)

# ---------------------------------------------------------
# 2. Load Memory-Mapped Video
# ---------------------------------------------------------
print(f"Mapping video from: {ram_path}")
if not os.path.exists(ram_path):
    raise FileNotFoundError(f"Could not find the file at {ram_path}")

# Load in read-only mode ('r') so we don't accidentally modify the data
video_mmap = np.memmap(
    ram_path, 
    dtype=np.float32, 
    mode='r', 
    shape=video_shape, 
    order=order
)

print(f"Successfully mapped array with shape: {video_mmap.shape}")

# ---------------------------------------------------------
# 3. Video Playback Loop (OpenCV Window)
# ---------------------------------------------------------
window_name = "CaImAn Memory Map Player (Press 'q' to Quit)"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

print("Starting playback. Click on the video window and press 'q' to stop.")

for f in range(frames):
    # 1. Pull a single frame: shape is (18, 1108)
    frame = video_mmap[f, :, :]
    
    # 2. Normalize data to 0-255 range for image display if it's float32 raw data
    f_min, f_max = frame.min(), frame.max()
    if f_max - f_min > 0:
        frame_display = ((frame - f_min) / (f_max - f_min) * 255).astype(np.uint8)
    else:
        frame_display = np.zeros_like(frame, dtype=np.uint8)
    # 3. STRETCH THE WINDOW VERTICALLY FOR VISUALIZATION
    # Native 18x1108 is too tiny to see. Let's scale the height up to 200 pixels.
    display_height = 18
    display_width = 1108
    frame_resized = cv2.resize(frame_display, (display_width, display_height), interpolation=cv2.INTER_NEAREST)
    
    # 4. Show the frame
    cv2.imshow(window_name, frame_resized)
    
    # 5. Frame rate control: wait 1 millisecond between frames
    # This also listens for keyboard input. If 'q' is pressed, break loop.
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Playback stopped by user.")
        break

# Clean up window and safely unmap reference
cv2.destroyAllWindows()
if hasattr(video_mmap, 'base') and hasattr(video_mmap.base, 'close'):
    video_mmap.base.close()
print("Playback window closed safely.")

Mapping video from: R:\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap
Successfully mapped array with shape: (36000, 18, 1108)
Starting playback. Click on the video window and press 'q' to stop.
Playback stopped by user.
Playback window closed safely.


In [87]:

###NEW SECTION FOR ROI COORDINATE EXTRACTION
cell_centers = [((y1 + y2) // 2, (x1 + x2) // 2) for (y1, x1, y2, x2) in Coords]
cell_centers = np.array(cell_centers)
print("Cell centers:", cell_centers)    
#display the cell centers on the image
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.imshow(img, cmap='gray') # Display the image
# ax.scatter(cell_centers[:, 1], cell_centers[:, 0], color='red') # Display the cell centers
# ax.set_title('Cell centers')    # Set the title of the plot
#plt.savefig(fname[:-4] + '_cell_centers.png', format='png', bbox_inches='tight', pad_inches=0)
 # Save the figure and close the plot     

# Save to a file
#save_path = fname[:-4] + '_cell_centers.npy'
#np.save(save_path, cell_centers)

#print(f"Cell centers saved to {save_path}")

#check if ROIS are empty and if so skip and save error
if ROIs.shape[0] == 0:
    print("No ROIs found.")
    raise ValueError("No ROIs detected, skipping further analysis for this trial.")
else:
    print(f"Found {ROIs.shape[0]} ROIs.")


cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)

##
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block

template_size = 0.008                         # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1 / 3                            # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'simple'                   # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 4                                 # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters
censor_size = 5                               # size of the censoring region around the ROI
min_width = 0                                 #minumum half peak-height width in ms
max_width = 9                                 #maximum half peak-height width in ms      
w_h_ratio = 2                                 #minumum ratio of height in %dF/F over half peak-height width in ms
                
correl_cutoff = 0.8
snr_thresh_display = 2

opts_dict={'fnames': ram_path,   #'fnames': fname_new,
        'ROIs': ROIs,
        'fr': fr,
        'index': index,
        'weights': weights,
        'min_width': min_width,
        'max_width': max_width,
        'w_h_ratio': w_h_ratio,
        'template_size': template_size,
        'context_size': context_size,
        'visualize_ROI': visualize_ROI,
        'hp_freq_pb': hp_freq_pb,
        'clip': clip,
        'threshold_method': threshold_method,
        'min_spikes':min_spikes,
        'pnorm': pnorm,
        'threshold': threshold,
        'do_plot':do_plot,
        'ridge_bg':ridge_bg,
        'sub_freq': sub_freq,
        'weight_update': weight_update,
        'n_iter': n_iter,
        'censor_size': censor_size}

#opts.change_params(params_dict=opts_dict)
opts = volparams(params_dict=opts_dict)

vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)

print("Running VOLPY fit...")
vpy.fit(n_processes=n_processes, dview=dview)
#takes a while to run
print("Done.")


Cell centers: [[  8 478]
 [  7 539]
 [  9 563]
 [  7 764]
 [  7 418]
 [  6 362]
 [  8 289]
 [  6 496]]
Found 8 ROIs.
Running VOLPY fit...
Starting VOLPY spike detection...
Done.


In [88]:

# Visualize spatial footprints and traces
#print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
# idx = np.where(vpy.estimates['locality'] > 0)[0]
# utils.view_components(vpy.estimates, img_corr, idx)


##

# Reconstructed movie
# flip_signal = True    
# mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=mc.mmap_file,
#                                         idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

##
vpy.estimates['ROIs'] = ROIs
vpy.estimates['Coords'] = Coords
# save_name = fname[:-4]+'_volpy'
# np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)

# print("Saved VOLPY estimates to:", save_name + '.npy')


print(vpy.estimates.keys())
print(len(vpy.estimates['spikes']))
#print(len(vpy.estimates['spikeTimes']))
print(vpy.estimates['snr']) 

#print length of each key's data:
for key in vpy.estimates.keys():
    print(f"{key}: {len(vpy.estimates[key])}")

#print number of neurons with snr > snr_thresh_display
high_snr_neurons = np.sum(vpy.estimates['snr'] > snr_thresh_display)
print(f"Number of neurons with SNR > {snr_thresh_display}: {high_snr_neurons}")


##
vpy = vpy.estimates
#vpy['spikes'] = np.array(vpy['spikes'], dtype=object)

# try:
num_frames = np.max(vpy['dFF'].shape)
dur = num_frames/640
vpy['snr_over_thresh'] = []

vpy['raster'] = np.zeros_like(vpy['dFF'])
vpy['firing_rate'] = np.zeros_like(vpy['dFF'])
vpy['unique_trace'] = []
vpy['cell_idxs'] = []

if vpy['spikes'].size > 0:

    for i in range(vpy['dFF'].shape[0]-1):
        vpy['raster'][i, vpy['spikes'][i]] = 1
        vpy['firing_rate'][i] = savgol_filter(np.convolve(vpy['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

    for i in range(len(vpy['ROIs'])):
        vpy['snr_over_thresh'].append(abs(vpy['snr'][i]) >= snr_thresh_display) #################################################################################################################
    print("SNR LIST", vpy['snr'])
    print("snr_over_thresh", vpy['snr_over_thresh'])
    print("Number of neurons with SNR > 0:", np.sum(vpy['snr_over_thresh']))

    if np.sum(vpy['snr_over_thresh']) > 0:
        to_remove = set()
        dFF = np.array(vpy['dFF']).astype(float)
        R = np.corrcoef(dFF)
        idx0, idx1 = np.where(np.triu(R, 1) > correl_cutoff) #################################################################################################################
        max_vals = np.max(dFF, axis=1)
        smaller = np.where(max_vals[idx0] < max_vals[idx1], idx0, idx1)
        to_remove.update(smaller.tolist())
        vpy['unique_trace'] = [True if x not in to_remove else False for x in range(len(vpy['ROIs']))]

    print(vpy['unique_trace'])
    print("Correl cutoff", correl_cutoff)
    print("There are", np.sum(vpy['unique_trace']), "unique traces after correlation filtering.")
    #print("And there were ", len(to_remove), "traces removed due to high correlation.")

    vpy['cell_idxs'] = []
    for cell in range(len(vpy['ROIs'])):
        if vpy['snr_over_thresh'][cell] and vpy['unique_trace'][cell]:
            vpy['cell_idxs'].append(cell)

    print("Final number of cells after SNR and correlation filtering:", len(vpy['cell_idxs']))
    print(vpy['cell_idxs'])
    print(len(vpy['cell_idxs']))
else:
    print("No spikes > threshold in this trial.")
    raise ValueError("No spikes detected, skipping further analysis for this trial.")

wheel_mat = os.path.dirname(fname) + '\\Wheel.mat'
if os.path.exists(wheel_mat):
    wheel=mat73.loadmat(wheel_mat)
    print("Loaded wheel data from:", wheel_mat)
else:
    print("No wheel data found at:", wheel_mat)
    wheel = None

#make figure
plotdata(vpy, dur, img, ROIs, fname, rootpath, unique_save_string, num_frames, mouseID, date, trialname, wheel)

print("Saving VOLPY data to MAT file...")
vpy['ROIs'] = ROIs
#vpy['rect'] = r['rois']
vpy['img'] = img
del vpy['rawROI']
#scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpy': vpy}, format='5', do_compression=True)

print("Converting data types for fast saving...")

# Add 1 to cell_idxs (MATLAB 1-based indexing)
if 'cell_idxs' in vpy:
    vpy['cell_idxs'] = [x + 1 for x in vpy['cell_idxs']]

# ---------------------------------------------------------
# Helper Function for Jagged Arrays
# ---------------------------------------------------------
def make_safe_object_array(data):
    """Bypasses NumPy broadcasting to safely package irregular lists/arrays."""
    if data is None:
        return np.empty(0, dtype=object)
    
    # If it's a single item, wrap it in a list to iterate
    if not isinstance(data, (list, tuple, np.ndarray)):
        data = [data]
        
    length = len(data)
    safe_arr = np.empty(length, dtype=object)
    for i in range(length):
        # If it's a sub-list (like spike times), ensure it's converted
        if isinstance(data[i], list):
            safe_arr[i] = np.array(data[i])
        else:
            safe_arr[i] = data[i]
    return safe_arr

# ---------------------------------------------------------
# 1. Process Float Conversions
# ---------------------------------------------------------
keys_to_convert_float = [
    't', 'ts', 't_rec', 't_sub', 'templates', 'snr', 
    'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 
    'raster', 'firing_rate'
]

for key in keys_to_convert_float:
    if key in vpy and vpy[key] is not None:
        try:
            vpy[key] = np.array(vpy[key], dtype=np.float32)
            print(f"  Converted '{key}' to float32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to float32. Forcing safe object array.")
            vpy[key] = make_safe_object_array(vpy[key])

# ---------------------------------------------------------
# 2. Process Integer Conversions
# ---------------------------------------------------------
keys_to_convert_int = ['num_spikes']

for key in keys_to_convert_int:
    if key in vpy and vpy[key] is not None:
        try:
            vpy[key] = np.array(vpy[key], dtype=np.int32)
            print(f"  Converted '{key}' to int32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to int32. Forcing safe object array.")
            vpy[key] = make_safe_object_array(vpy[key])

# ---------------------------------------------------------
# 3. Process Jagged Object Arrays (mean_im, spikes, etc.)
# ---------------------------------------------------------
jagged_keys = ['mean_im', 'cell_n', 'polarity', 'spikes', 'low_spikes']

for key in jagged_keys:
    if key in vpy and vpy[key] is not None:
        try:
            # Check if low_spikes happens to be a clean boolean array first
            if key == 'low_spikes' and isinstance(vpy[key], np.ndarray) and vpy[key].dtype == bool:
                continue # Leave it alone, it's fine
                
            vpy[key] = make_safe_object_array(vpy[key])
            print(f"  Safely packaged '{key}' as a NumPy object array.")
        except Exception as e:
            print(f"  Warning: Could not process '{key}': {e}")

print("Data type conversion complete.")

def clean_none(data, name="root"):
    if data is None:
        # Print the name of the key that has the None value
        print(f"Replacing None with [] at: {name}")
        return [] 
    
    elif isinstance(data, dict):
        # Recursively clean each key, passing the key name down for the print statement
        return {k: clean_none(v, name=f"{name} -> {k}") for k, v in data.items()}
    
    elif isinstance(data, list):
        # Recursively clean each list item, passing the index for the print statement
        return [clean_none(v, name=f"{name}[{i}]") for i, v in enumerate(data)]
    
    return data

vpy = clean_none(vpy)

print("Data type conversion complete.")

scipy.io.savemat(rootpath + unique_save_string + '.mat', {'vpy': vpy}, format='5', do_compression=True)
print("Saved VOLPY data to:", fname[:-4] + '_volpy.mat')


# vpy.estimates['params'] = opts
# save_name = f'volpy_{os.path.split(fnames)[1][:-5]}_{threshold_method}'
# np.save(fnames[:-4] + '_volpy.npy', vpy.estimates)

del vpy
# % STOP CLUSTER and clean up log files

log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)
# except ValueError as e:
#     print(e)
#     print("No volpy data was saved")

# Cleanup R:/ drive
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")

#Append new row to MASTERLOG
today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
print(today_str)
new_row = [version, today_str, unique_save_string]

with open(log_csv_path, mode='a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(new_row)
print(f"Added new row to MasterAnalysisLOG.csv: {new_row}")



    # print(f"ERROR processing {fname}: {e}")
    # #Append new row to MASTERLOG
    # today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
    # new_row = [version, today_str, unique_save_string, e]

    # with open(log_csv_path, mode='a', newline='') as f:
    #     writer = csv.writer(f)
    #     writer.writerow(new_row)
    # print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")


dict_keys(['rawROI', 'mean_im', 'cell_n', 't', 'ts', 't_rec', 't_sub', 'spikes', 'low_spikes', 'num_spikes', 'templates', 'snr', 'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 'polarity', 'ROIs', 'Coords'])
8
[0 3.417211105487812 3.482673047031307 3.3577875543696885
 3.3167412595107524 3.511951892969774 1.271686175579562 0]
rawROI: 8
mean_im: 8
cell_n: 8
t: 8
ts: 8
t_rec: 8
t_sub: 8
spikes: 8
low_spikes: 8
num_spikes: 8
templates: 8
snr: 8
thresh: 8
weights: 8
locality: 8
context_coord: 8
F0: 8
dFF: 8
polarity: 8
ROIs: 8
Coords: 8
Number of neurons with SNR > 2: 5
SNR LIST [0 3.417211105487812 3.482673047031307 3.3577875543696885
 3.3167412595107524 3.511951892969774 1.271686175579562 0]
snr_over_thresh [False, True, True, True, True, True, False, False]
Number of neurons with SNR > 0: 5
[False, True, True, True, True, True, True, True]
Correl cutoff 0.8
There are 7 unique traces after correlation filtering.
Final number of cells after SNR and correlation filtering: 5
[